In [1]:
# =========================================================
# STEP 0: IMPORTS
# =========================================================
#import os
#import json
#import numpy as np
#import cv2
#import zipfile
#from tqdm import tqdm

#print("Starting mask generation...")


# =========================================================
# STEP 1: PATHS
# =========================================================
#BASE_PATH = "/kaggle/input/datasets/kunalnarang47/deepfashion-redwing/deepfashion2_pruned"
#OUTPUT_PATH = "/kaggle/working/unet_masks"   # custom folder
#ZIP_PATH = "/kaggle/working/unet_masks_fixed.zip"  # custom zip name


# =========================================================
# STEP 2: CATEGORY MAP
# =========================================================
#CATEGORY_MAP = {
#    1: 1,
#    2: 2,
#    7: 3,
#    8: 4,
#    9: 5
#}


# =========================================================
# STEP 3: MASK FUNCTION
# =========================================================
#def create_mask(json_path, height, width):
#    mask = np.zeros((height, width), dtype=np.uint8)

#    with open(json_path, 'r') as f:
  #      data = json.load(f)

 #   for key in data:
  #      item = data[key]

 #       cid = item.get("category_id")
  #      segmentation = item.get("segmentation", [])
#
 #       if cid not in CATEGORY_MAP:
 #           continue

  #      class_id = CATEGORY_MAP[cid]

  #      for seg in segmentation:
  #          if not isinstance(seg, list) or len(seg) < 6:
   #             continue
#
  #          pts = np.array(seg, dtype=np.int32).reshape(-1, 2)

 #           if pts.shape[0] < 3:
#                continue
#
 #           cv2.fillPoly(mask, [pts], class_id)

 #   return mask


# =========================================================
# STEP 4: PROCESS FUNCTION (LOW LOG)
# =========================================================
#def process_split(split):
#    print(f"\nProcessing {split}...")

 #   img_dir = os.path.join(BASE_PATH, split, "image")
#    anno_dir = os.path.join(BASE_PATH, split, "annos")
  #  save_dir = os.path.join(OUTPUT_PATH, split)

#    os.makedirs(save_dir, exist_ok=True)

#    files = os.listdir(anno_dir)
#    saved_count = 0

    # 🔥 No tqdm spam → simple loop
#    for i, file in enumerate(files):
 #       if not file.endswith(".json"):
 #           continue

   #     json_path = os.path.join(anno_dir, file)
 #       img_name = file.replace(".json", ".jpg")
#        img_path = os.path.join(img_dir, img_name)

#        if not os.path.exists(img_path):
  #          continue

#        img = cv2.imread(img_path)
#        h, w = img.shape[:2]
#
#        mask = create_mask(json_path, h, w)

#        if np.sum(mask) == 0:
 #           continue

#       save_path = os.path.join(save_dir, img_name.replace(".jpg", ".png"))
#        cv2.imwrite(save_path, mask)

 #       saved_count += 1

        # 🔥 Minimal progress print every 2000 images
#        if i % 2000 == 0:
#            print(f"{split}: processed {i}/{len(files)}")

 #   print(f"{split}: {saved_count} masks saved")


# =========================================================
# STEP 5: RUN
# =========================================================
#process_split("train")
#process_split("validation")

#print("\nMask generation done")


# =========================================================
# STEP 6: ZIP OUTPUT (CUSTOM NAME)
# =========================================================
#print("Zipping dataset...")

#with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zipf:
#    for root, _, files in os.walk(OUTPUT_PATH):
#        for file in files:
 #           file_path = os.path.join(root, file)
#            arcname = os.path.relpath(file_path, OUTPUT_PATH)
 #           zipf.write(file_path, arcname)

#print(f"✅ ZIP CREATED: {ZIP_PATH}")

In [2]:
# =========================================================
# IMPORTS
# =========================================================
import os
import zipfile
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torch.nn as nn
import torch.nn.functional as F

print("Starting U-Net pipeline...")


# =========================================================
# STEP 1: UNZIP DATASET
# =========================================================
ZIP_PATH = "/kaggle/input/notebooks/redwing47/unet-train/_output_.zip"
EXTRACT_PATH = "/kaggle/working"

print("\nExtracting dataset...")

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Extraction done")


# =========================================================
# PATHS
# =========================================================
IMG_BASE = "/kaggle/input/datasets/kunalnarang47/deepfashion-redwing/deepfashion2_pruned/train/image"
MASK_BASE = "/kaggle/working/unet_masks/train"


# =========================================================
# DATASET
# =========================================================
class UNetDataset(Dataset):
    def __init__(self, img_dir, mask_dir, size=256):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.files = os.listdir(mask_dir)
        self.size = size

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        name = self.files[idx]

        img_path = os.path.join(self.img_dir, name.replace(".png", ".jpg"))
        mask_path = os.path.join(self.mask_dir, name)

        img = cv2.imread(img_path)
        img = cv2.resize(img, (self.size, self.size))

        mask = cv2.imread(mask_path, 0)
        mask = cv2.resize(mask, (self.size, self.size), interpolation=cv2.INTER_NEAREST)

        img = img / 255.0
        img = np.transpose(img, (2,0,1))

        return torch.tensor(img, dtype=torch.float32), torch.tensor(mask, dtype=torch.long)


# =========================================================
# MODEL
# =========================================================
class ResNetUNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()

        base = models.resnet18(pretrained=pretrained)

        self.encoder = nn.Sequential(*list(base.children())[:-2])

        self.up1 = nn.ConvTranspose2d(512,256,2,2)
        self.up2 = nn.ConvTranspose2d(256,128,2,2)
        self.up3 = nn.ConvTranspose2d(128,64,2,2)
        self.up4 = nn.ConvTranspose2d(64,64,2,2)

        self.out = nn.Conv2d(64,6,1)

    def forward(self, x):
        x = self.encoder(x)

        x = self.up1(x)
        x = self.up2(x)
        x = self.up3(x)
        x = self.up4(x)

        return self.out(x)


# =========================================================
# TRAIN FUNCTION
# =========================================================
def train_model(model, loader, epochs, name):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()

    print(f"\n{name} training started")

    for epoch in range(epochs):
        total_loss = 0
        model.train()

        print(f"{name} Epoch {epoch+1}/{epochs}")

        for i, (imgs, masks) in enumerate(loader):
            imgs = imgs.to(device)
            masks = masks.to(device)

            outputs = model(imgs)

            # Fix size mismatch
            outputs = F.interpolate(outputs, size=masks.shape[1:], mode='bilinear', align_corners=False)

            loss = criterion(outputs, masks)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            # Minimal logs
            if i % (len(loader)//10 + 1) == 0:
                print(f"  {int(i/len(loader)*100)}% done")

        print(f"Loss: {total_loss:.4f}")

    torch.save(model.state_dict(), f"/kaggle/working/{name}.pth")
    print(f"{name} saved")


# =========================================================
# DATALOADER
# =========================================================
train_loader = DataLoader(
    UNetDataset(IMG_BASE, MASK_BASE),
    batch_size=8,
    shuffle=True,
    num_workers=2
)

print("Dataset ready")


# =========================================================
# TRAIN TRANSFER
# =========================================================
transfer_model = ResNetUNet(pretrained=True)
train_model(transfer_model, train_loader, 10, "unet_transfer")


# =========================================================
# TRAIN SCRATCH
# =========================================================
scratch_model = ResNetUNet(pretrained=False)
train_model(scratch_model, train_loader, 10, "unet_scratch")


print("\n✅ U-NET TRAINING COMPLETE")

Starting U-Net pipeline...

Extracting dataset...
Extraction done
Dataset ready


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 183MB/s]



unet_transfer training started
unet_transfer Epoch 1/10
  0% done
  10% done
  20% done
  30% done
  40% done
  50% done
  60% done
  70% done
  80% done
  90% done
Loss: 5654.7171
unet_transfer Epoch 2/10
  0% done
  10% done
  20% done
  30% done
  40% done
  50% done
  60% done
  70% done
  80% done
  90% done
Loss: 3822.7557
unet_transfer Epoch 3/10
  0% done
  10% done
  20% done
  30% done
  40% done
  50% done
  60% done
  70% done
  80% done
  90% done
Loss: 3189.2764
unet_transfer Epoch 4/10
  0% done
  10% done
  20% done
  30% done
  40% done
  50% done
  60% done
  70% done
  80% done
  90% done
Loss: 2730.3735
unet_transfer Epoch 5/10
  0% done
  10% done
  20% done
  30% done
  40% done
  50% done
  60% done
  70% done
  80% done
  90% done
Loss: 2354.8579
unet_transfer Epoch 6/10
  0% done
  10% done
  20% done
  30% done
  40% done
  50% done
  60% done
  70% done
  80% done
  90% done
Loss: 2078.7385
unet_transfer Epoch 7/10
  0% done
  10% done
  20% done
  30% done


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


  0% done
  10% done
  20% done
  30% done
  40% done
  50% done
  60% done
  70% done
  80% done
  90% done
Loss: 8420.9008
unet_scratch Epoch 2/10
  0% done
  10% done
  20% done
  30% done
  40% done
  50% done
  60% done
  70% done
  80% done
  90% done
Loss: 5912.3959
unet_scratch Epoch 3/10
  0% done
  10% done
  20% done
  30% done
  40% done
  50% done
  60% done
  70% done
  80% done
  90% done
Loss: 5037.9762
unet_scratch Epoch 4/10
  0% done
  10% done
  20% done
  30% done
  40% done
  50% done
  60% done
  70% done
  80% done
  90% done
Loss: 4470.0010
unet_scratch Epoch 5/10
  0% done
  10% done
  20% done
  30% done
  40% done
  50% done
  60% done
  70% done
  80% done
  90% done
Loss: 4011.9770
unet_scratch Epoch 6/10
  0% done
  10% done
  20% done
  30% done
  40% done
  50% done
  60% done
  70% done
  80% done
  90% done
Loss: 3598.3848
unet_scratch Epoch 7/10
  0% done
  10% done
  20% done
  30% done
  40% done
  50% done
  60% done
  70% done
  80% done
  90% do